# U14 — Supervised Learning Foundations: Lab

### Real-world brief: predicting machine failure (predictive maintenance)

A factory wants to flag machines likely to **fail**, so maintenance can act before a costly breakdown. You'll learn the supervised-learning workflow on this imbalanced binary-classification problem: frame it, fit/predict/score, **compare model families with the same API**, evaluate with the right metrics, and cross-validate.

**Resource provided:** `machine_maintenance.csv` (one row per machine, target = `machine_failure`). Keep it beside this notebook (upload it in Colab).

_Phase D — Modelling._

#objectives

Frame a supervised classification problem (X, y, imbalance)

Use the fit / predict / score pattern that every sklearn model shares

Compare model families (logistic regression, tree, k-NN, random forest)

Evaluate honestly: accuracy vs precision / recall / F1 / ROC-AUC

Cross-validate, and see how a hyperparameter drives over/underfitting

#how to use this lab

Worked demos teach the pattern; 🧪 LAB EXERCISE cells are real tasks — replace `# YOUR CODE HERE`. Run top to bottom with Shift+Enter.

In [ ]:
# === SETUP: load the provided file (regenerate it if missing) ===
import os
import numpy as np
import pandas as pd


def build_maintenance(csv_path="machine_maintenance.csv", seed=14, verbose=False):
    """Predictive maintenance: predict whether a machine will FAIL from its
    operating conditions. A realistic, imbalanced binary-classification problem —
    perfect for the full supervised workflow and comparing model families.

    Features:
      machine_type        product quality variant (L / M / H)
      air_temperature_k    ambient temperature (K)
      process_temperature_k process temperature (K)
      rotational_speed_rpm spindle speed (rpm)
      torque_nm            torque (Nm)
      tool_wear_min        cumulative tool wear (minutes)
    Target:
      machine_failure      1 = failed, 0 = healthy
    """
    rng = np.random.default_rng(seed)
    N = 3000
    mtype = rng.choice(["L", "M", "H"], N, p=[0.50, 0.30, 0.20])
    air = rng.normal(300, 2.0, N)
    process = air + 10 + rng.normal(0, 1.0, N)
    speed = np.clip(rng.normal(1540, 170, N), 1168, 2886).round().astype(int)
    torque = np.clip(rng.normal(40, 10, N), 3, 77).round(1)
    wear = rng.uniform(0, 253, N).round().astype(int)
    type_eff = np.select([mtype == "L", mtype == "M", mtype == "H"], [0.6, 0.0, -0.5])

    # failure risk: worn tools, high torque, an overstrain interaction, and poor heat dissipation
    heat_bad = ((process - air) < 8.6).astype(float)
    z = (-5.2
         + 0.012 * wear
         + 0.06 * (torque - 40)
         + 4.0e-4 * wear * (torque - 40)
         + 0.7 * heat_bad
         + type_eff)
    p = 1 / (1 + np.exp(-z))
    failure = (rng.random(N) < p).astype(int)

    df = pd.DataFrame({
        "machine_type": mtype,
        "air_temperature_k": air.round(1),
        "process_temperature_k": process.round(1),
        "rotational_speed_rpm": speed,
        "torque_nm": torque,
        "tool_wear_min": wear,
        "machine_failure": failure,
    })
    df.to_csv(csv_path, index=False)
    if verbose:
        print("maintenance:", df.shape)
        print("failure rate:", round(df.machine_failure.mean(), 3))
        print("failures by type:\n",
              df.groupby("machine_type")["machine_failure"].mean().round(3).to_string())
    return df

if not os.path.exists('machine_maintenance.csv'):
    build_maintenance(); print('Generated dataset file.')
else:
    print('Found the provided dataset file.')

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
df = pd.read_csv('machine_maintenance.csv')
print('shape:', df.shape)
print('failure rate:', round(df['machine_failure'].mean(), 3))
df.head(3)

#1. Frame the problem

In [ ]:
# -----------------------------------------------------------
# 🔹 1A. TARGET BALANCE + FEATURES / TARGET
# -----------------------------------------------------------
ax = df['machine_failure'].value_counts().plot(kind='bar', color=['#2D6A9F', '#C0392B'], figsize=(4.5, 3))
ax.set_xticklabels(['healthy (0)', 'failed (1)'], rotation=0); ax.set_ylabel('count')
ax.set_title('Imbalanced target'); plt.tight_layout(); plt.show()

y = df['machine_failure']
X = df.drop(columns='machine_failure')
print('X:', X.shape, '| classification, 2 classes')
print('Because failures are rare, ACCURACY will be misleading — watch recall/F1/AUC.')

#2. The fit / predict / score pattern

In [ ]:
# -----------------------------------------------------------
# 🔹 2A. PREPROCESS (encode + scale) + A FIRST MODEL
# -----------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

num = X.select_dtypes('number').columns.tolist()
cat = ['machine_type']
pre = ColumnTransformer([('num', StandardScaler(), num),
                         ('cat', OneHotEncoder(), cat)])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

clf = Pipeline([('prep', pre), ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))])
clf.fit(X_train, y_train)                 # FIT — learn from training data
preds = clf.predict(X_test)               # PREDICT — apply to unseen data
print('score (accuracy):', round(clf.score(X_test, y_test), 3))

#### 🧪 EXERCISE 2 — Accuracy can lie
1. Compute the accuracy of a 'dummy' rule that always predicts **healthy (0)** (hint: `1 - y_test.mean()`).
2. Compare it with the model's accuracy above.
3. In a comment, explain why high accuracy alone doesn't prove the model is useful here.

In [ ]:
# 1. always-healthy accuracy
# YOUR CODE HERE

# 2-3. compare & explain: ...   (comment)

#3. Same API, different models

Every sklearn classifier shares `fit` / `predict`. Swapping families is a one-line change.

In [ ]:
# -----------------------------------------------------------
# 🔹 3A. TRAIN FOUR FAMILIES, COMPARE ON TEST
# -----------------------------------------------------------
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score

models = {
    'LogReg': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'DecisionTree': DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=0),
    'kNN': KNeighborsClassifier(n_neighbors=15),
    'RandomForest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=0),
}
rows = []
for name, m in models.items():
    pipe = Pipeline([('prep', pre), ('model', m)])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    rows.append({'model': name,
                 'F1': f1_score(y_test, pipe.predict(X_test)),
                 'ROC_AUC': roc_auc_score(y_test, proba)})
results = pd.DataFrame(rows).set_index('model').round(3)
print(results)

#### 🧪 EXERCISE 3 — Chart the comparison
1. Make a grouped bar chart of F1 and ROC-AUC for the four models (`results.plot(kind='bar')`).
2. In a comment, name the best model on this data and by how much it beats logistic regression.

In [ ]:
# 1. bar chart of the comparison
# YOUR CODE HERE

# 2. best model & margin: ...   (comment)

#4. Classification metrics that matter

In [ ]:
# -----------------------------------------------------------
# 🔹 4A. CONFUSION MATRIX + FULL REPORT (random forest)
# -----------------------------------------------------------
from sklearn.metrics import confusion_matrix, classification_report
rf = Pipeline([('prep', pre), ('model', RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=0))]).fit(X_train, y_train)
cm = confusion_matrix(y_test, rf.predict(X_test))
fig, ax = plt.subplots(figsize=(4, 3.4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['healthy', 'failed'], yticklabels=['healthy', 'failed'], ax=ax)
ax.set_xlabel('predicted'); ax.set_ylabel('actual'); ax.set_title('Confusion matrix')
plt.tight_layout(); plt.show()
print(classification_report(y_test, rf.predict(X_test), digits=3))

#### 🧪 EXERCISE 4 — Precision vs recall trade-off
For predictive maintenance, missing a real failure (a false negative) is usually costlier than a false alarm.
1. From the report, read off precision and recall for the **failed** class.
2. In a comment, say which metric you'd prioritise here and why.
3. Bonus: re-predict using a lower probability threshold (e.g. `proba > 0.3`) and show recall rises.

In [ ]:
proba_rf = rf.predict_proba(X_test)[:, 1]
# 1-2. read precision/recall for the failed class; which matters more? ...   (comment)

# 3. lower the threshold to 0.3 and recompute recall
# YOUR CODE HERE

#5. Cross-validation for a stable comparison

In [ ]:
# -----------------------------------------------------------
# 🔹 5A. 5-FOLD ROC-AUC (mean +/- std) PER MODEL
# -----------------------------------------------------------
from sklearn.model_selection import cross_val_score
for name, m in models.items():
    pipe = Pipeline([('prep', pre), ('model', m)])
    s = cross_val_score(pipe, X, y, cv=5, scoring='roc_auc')
    print(f'{name:14s} ROC-AUC {s.mean():.3f} +/- {s.std():.3f}')

#### 🧪 EXERCISE 5 — One split can mislead
1. Run `cross_val_score` with `scoring='f1'` for the random forest and print each fold's score.
2. In a comment, note the spread between folds — and why a single train/test split could over- or under-state performance.

In [ ]:
# 1. per-fold F1 for the random forest
# YOUR CODE HERE

# 2. fold spread & why CV matters: ...   (comment)

#6. Parameters vs hyperparameters

In [ ]:
# -----------------------------------------------------------
# 🔹 6A. A HYPERPARAMETER (tree depth) DRIVES OVER/UNDERFIT
# -----------------------------------------------------------
depths = range(1, 16)
tr, te = [], []
for d in depths:
    p = Pipeline([('prep', pre), ('model', DecisionTreeClassifier(
        max_depth=d, class_weight='balanced', random_state=0))]).fit(X_train, y_train)
    tr.append(f1_score(y_train, p.predict(X_train)))
    te.append(f1_score(y_test, p.predict(X_test)))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(depths), tr, 'o-', label='train F1', color='#2D6A9F')
ax.plot(list(depths), te, 'o-', label='test F1', color='#C0392B')
ax.set_xlabel('max_depth (a hyperparameter)'); ax.set_ylabel('F1'); ax.legend()
ax.set_title('Deep trees overfit: train F1 -> 1, test F1 drops'); plt.tight_layout(); plt.show()

#### 🧪 EXERCISE 6 — Pick the depth
1. Find the `max_depth` with the highest **test** F1.
2. In a comment, distinguish a *parameter* (learned in fit) from a *hyperparameter* (you chose `max_depth`), using this tree as the example.

In [ ]:
# 1. best max_depth by test F1
# YOUR CODE HERE

# 2. parameter vs hyperparameter here: ...   (comment)

#📘 Summary

| Step | Tool | Lesson |
| ---- | ---- | ------ |
| Frame | X, y, class balance | rare positives -> accuracy misleads |
| fit/predict/score | one API | the core of every sklearn model |
| Compare families | loop over models | swapping is a one-line change |
| Metrics | precision/recall/F1/AUC | match the metric to the cost of errors |
| Cross-validation | cross_val_score | a stable, fold-averaged estimate |
| Hyperparameters | tree depth | you tune these; fit learns the parameters |

**Core lesson:** supervised learning is a disciplined loop — frame, fit, evaluate honestly on unseen data with the right metric, and compare.

**Next — U15 Regression (Part 1):** open up one model family in depth, starting with linear regression.